In [3]:
%pip install -q google-genai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import getpass 
import json

import joblib #para cargar el modelo
import pandas as pd
from google import genai
from google.genai import types

In [ ]:
api_key = getpass.getpass("Pegá tu API key: ")
client = genai.Client(api_key=api_key) #crea la conexion con gemini
print("Cliente de Gemini listo")

Cliente de Gemini listo


In [ ]:
def prioridad_modelo(nombre):
    puntaje = 100
    if "flash" in nombre: puntaje -= 50
    if "pro" in nombre: puntaje -= 20
    if "preview" in nombre or "latest" in nombre: puntaje += 30
    return puntaje

excluir = ["tts", "image", "embedding", "live", "robotics"]
candidatos = [
    m.name for m in client.models.list()
    if "gemini" in m.name
    and not any(x in m.name for x in excluir)
    and "generateContent" in (m.supported_actions or []) #si acepta generate content
]
candidatos = sorted(candidatos, key=prioridad_modelo)

MODELO = None
for candidato in candidatos:
    try:
        client.models.generate_content(model=candidato, contents="Respondé OK")
        MODELO = candidato
        break
    except Exception as error:
        print("No disponible:", candidato, "-", str(error)[:80])

print("Modelo elegido:", MODELO)

No disponible: models/gemini-2.5-flash - 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-
No disponible: models/gemini-2.5-flash-lite - 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-
Modelo elegido: models/gemini-3.1-flash-lite


In [ ]:
df = pd.read_csv("../Datos/dataset_features.csv", encoding="utf-8-sig", low_memory=False) #lee el data set
df["fecha_dt"] = pd.to_datetime(df["fecha_dt"]) #la fecha
df = df.sort_values(["fecha_dt", "votacion_id"]) #ordena

ultima = df["votacion_id"].iloc[-1] #la ultima votacion
en_ejercicio = df.loc[df["votacion_id"] == ultima, "diputado"] #diputados en ejercicio

print(len(en_ejercicio))

256


In [ ]:
grupo = df[df["diputado"].isin(en_ejercicio)].groupby("diputado") #agrupo por los diputados en ejercicio

perfiles = pd.DataFrame({
    "tasa_afirmativos_previos": grupo["es_afirmativo"].mean() * 100,
    "tasa_negativos_previos": grupo["es_negativo"].mean() * 100,
    "tasa_abstenciones_previas": grupo["es_abstencion"].mean() * 100,
    "tasa_ausencias_previas": grupo["es_ausente"].mean() * 100, #el promedio historico de cada diputado
    "tasa_afirmativos_recientes": grupo["es_afirmativo"].apply(lambda s: s.tail(50).mean() * 100),
    "tasa_ausencias_recientes": grupo["es_ausente"].apply(lambda s: s.tail(50).mean() * 100), #tasa sus ultimos 50 votos
    "total_votos_previos": grupo.size(),
    "macrocoalicion": grupo["macrocoalicion"].last(),
    "provincia": grupo["provincia"].last(), #su macrocoalicion y provincias actuales
}).reset_index()

perfiles["es_oficialismo"] = (perfiles["macrocoalicion"] == "LLA").astype(int) #oficialismo hoy

perfiles["macrocoalicion"].value_counts()

macrocoalicion
PERONISMO       94
LLA             94
PROVINCIALES    37
PRO             14
RADICALISMO      8
OTROS            5
IZQUIERDA        4
Name: count, dtype: int64

In [ ]:
modelo = joblib.load("../modelo_random_forest.pkl") #carga el modelo entrenado

In [10]:
prompt_ley = """Sos un especialista en el Congreso argentino. Te paso el título o resumen de un proyecto que se trata en la Cámara de Diputados.

Devolvé un JSON con tres claves:
- "origen": quién presentó el proyecto. "PE" si lo envió el Poder Ejecutivo, "JGM" si lo envió la Jefatura de Gabinete, "S" si viene del Senado (en revisión), "D" si lo presentó un diputado, "SIN_DATO" si no se puede saber.
- "es_mocion": 1 si es un trámite (moción de orden, tratamiento sobre tablas, vuelta a comisión, cambio del plan de labor), 0 si se vota la ley o sus artículos.
- "resumen": una oración que diga de qué trata.
"""

In [ ]:
def analizar_ley(texto):
    respuesta = client.models.generate_content(
        model=MODELO,
        contents=[prompt_ley, texto],
        config=types.GenerateContentConfig(response_mime_type="application/json"),
    ) #le pasa el prompt y el texto. Lo obliga a responder en json
    ley = json.loads(respuesta.text) #convierte la lista en diccionario
    if isinstance(ley, list): 
        ley = ley[0]

    origen = ley.get("origen", "SIN_DATO")
    if origen not in ["PE", "JGM", "S", "D"]:
        origen = "SIN_DATO" #si gemini invento el origrn pone sin dato
    es_mocion = int(ley.get("es_mocion", 0)) #lee si es mocion y lo pasa a numero

    print("Gemini entendió:", ley.get("resumen", ""))
    print("Origen:", origen, "| Moción:", es_mocion) #muestra lo que entendio gemini

    hoy = pd.Timestamp.today() #fecha de hoy
    entrada = perfiles.copy() #copia los perfiles 
    entrada["origen"] = origen
    entrada["es_mocion"] = es_mocion
    entrada["anio"] = hoy.year
    entrada["mes"] = hoy.month #les agrega los datos del proyecto y la fecha

    entrada["probabilidad"] = modelo.predict_proba(entrada)[:, 1] #probabilidad de afirmativo de cada diputado
    entrada["a_favor"] = (entrada["probabilidad"] > 0.5).astype(int) # a favor si es mayor de 0,5

    por_bloque = entrada.groupby("macrocoalicion").agg(
        diputados=("diputado", "size"),
        a_favor=("a_favor", "sum"),
    ).sort_values("diputados", ascending=False) #resume por macrocoalicion

    total = entrada["a_favor"].sum()
    print("Votos a favor esperados:", total, "de", len(entrada))
    print("Mayoría absoluta (129):", "SÍ" if total >= 129 else "NO")

    return entrada[["diputado", "macrocoalicion", "provincia", "probabilidad"]].sort_values("probabilidad")

In [12]:
analizar_ley("Proyecto de ley del Poder Ejecutivo de modernización laboral: modifica la Ley de Contrato de Trabajo y el régimen de indemnizaciones.")

Gemini entendió: El proyecto propone modernizar la legislación laboral mediante reformas en la Ley de Contrato de Trabajo y en el esquema vigente de indemnizaciones.
Origen: PE | Moción: 0
Votos a favor esperados: 168 de 256
Mayoría absoluta (129): SÍ


,diputado,macrocoalicion,provincia,probabilidad
189,"PICHETTO, MIGUEL ANGEL",PERONISMO,Buenos Aires,0.191342
121,"LANESAN SANCHO, MOIRA",PERONISMO,Santa Cruz,0.210455
35,"BREGMAN, MYRIAM",IZQUIERDA,C.A.B.A.,0.247119
173,"PAGANO, MARCELA MARINA",OTROS,Buenos Aires,0.248490
53,"CISNEROS, CARLOS",PERONISMO,Tucumán,0.253562
...,...,...,...,...
176,"PAREJA, SEBASTIAN",LLA,Buenos Aires,0.856830
237,"URIEN, HERNAN",LLA,Buenos Aires,0.856830
200,"REICHARDT, KAREN",LLA,Buenos Aires,0.856830
232,"TORRES, RUBEN DARIO",LLA,Buenos Aires,0.856830
